# RAISE HTML5 Semantification Demo

Owner: Siddharth Tripathi  
Repository: `semanticClimate/RAISE`  
Branch: `siddharth-semantification`  
Module folder: `RAISE_HTML5_Semantification`

Task: convert prepared annual-report block JSON into clean, traceable semantic HTML5.

Upload `target_blocks.json`. Do not upload a PDF; PDF parsing and filtering happen before this module.

In [ ]:
from google.colab import files

uploaded = files.upload()
input_filename = next(iter(uploaded))
print(f"Uploaded JSON file: {input_filename}")

In [ ]:
!pip install -q beautifulsoup4 lxml jinja2 pydantic jsonschema typer rich

In [ ]:
# Install from the exact RAISE monorepo branch and module folder when opened from GitHub Colab.
# If installation is unavailable during a live demo, the notebook continues with its self-contained engine.

import subprocess
import sys

GITHUB_PACKAGE = "git+https://github.com/semanticClimate/RAISE.git@siddharth-semantification#subdirectory=RAISE_HTML5_Semantification"

try:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        GITHUB_PACKAGE,
    ])
    print("Installed RAISE_HTML5_Semantification from semanticClimate/RAISE.")
except Exception as exc:
    print("GitHub package install was not completed; using the self-contained notebook engine.")
    print(str(exc))

In [ ]:
import os
import platform

runtime_info = {
    "platform": platform.platform(),
    "colab_tpu_addr": os.environ.get("COLAB_TPU_ADDR", ""),
}

if runtime_info["colab_tpu_addr"]:
    print(f"TPU runtime detected: {runtime_info['colab_tpu_addr']}")
else:
    print("No TPU runtime detected. HTML5 semantification is CPU-safe and deterministic.")

runtime_info

## Semantification Engine

The Colab workflow below mirrors the repository workflow: load structured blocks, classify headings/lists/tables, write `report.html`, and write `validation_report.json`.

In [ ]:
import html
import json
import re
from pathlib import Path
from statistics import median

from bs4 import BeautifulSoup

SECTION_KEYWORDS = {
    "department", "research", "publications", "grants", "patents", "faculty",
    "awards", "collaboration", "workshop", "conference", "outreach", "activities",
}
BLOCK_CONTAINER_KEYS = (
    "blocks", "target_blocks", "content_blocks", "pages", "items",
    "elements", "paragraphs", "lines", "records",
)
TEXT_KEYS = {"text", "content", "block_text", "raw_text", "value", "line", "paragraph"}
TABLE_KEYS = {"rows", "table", "table_rows", "tableRows", "table_data", "tableData", "cells"}
ALIASES = {
    "block_id": ("block_id", "id", "blockId", "blockID", "uid", "uuid"),
    "page_number": ("page_number", "page", "page_num", "pageNumber", "page_no"),
    "text": ("text", "content", "block_text", "raw_text", "value", "line", "paragraph"),
    "bbox": ("bbox", "bounding_box", "bounds", "box"),
    "font_size": ("font_size", "fontsize", "fontSize", "size"),
    "font_name": ("font_name", "font", "fontName", "font_family"),
    "is_bold": ("is_bold", "bold", "isBold"),
    "is_italic": ("is_italic", "italic", "isItalic"),
    "reading_order": ("reading_order", "order", "readingOrder", "sequence", "index"),
    "block_type_guess": ("block_type_guess", "type", "block_type", "label", "category", "role"),
    "confidence": ("confidence", "score", "probability", "ocr_confidence"),
    "section_hint": ("section_hint", "section", "heading_hint", "sectionHint"),
    "source_file": ("source_file", "source", "file", "filename", "document"),
    "rows": ("rows", "table_rows", "tableRows", "cells"),
    "table": ("table", "table_data", "tableData"),
}

def looks_like_block(value):
    if not isinstance(value, dict):
        return False
    keys = set(value)
    metadata = {"block_id", "id", "page_number", "page", "bbox", "font_size", "type", "block_type_guess"}
    return bool(keys & TEXT_KEYS or keys & TABLE_KEYS or keys & metadata)

def candidate_score(items):
    return sum(1 for item in items if looks_like_block(item))

def find_best_block_list(raw):
    candidates = []
    def visit(value):
        if isinstance(value, list):
            if candidate_score(value):
                candidates.append(value)
            for item in value:
                visit(item)
        elif isinstance(value, dict):
            for key in BLOCK_CONTAINER_KEYS:
                nested = value.get(key)
                if isinstance(nested, list) and candidate_score(nested):
                    candidates.append(nested)
            for nested in value.values():
                visit(nested)
    visit(raw)
    if not candidates:
        return None
    return [item for item in max(candidates, key=lambda items: (candidate_score(items), len(items))) if isinstance(item, dict)]

def extract_blocks(raw):
    if isinstance(raw, list) and candidate_score(raw):
        return [item for item in raw if isinstance(item, dict)]
    if isinstance(raw, dict):
        for key in BLOCK_CONTAINER_KEYS:
            nested = raw.get(key)
            if isinstance(nested, list) and candidate_score(nested):
                return [item for item in nested if isinstance(item, dict)]
        if looks_like_block(raw):
            return [raw]
    inferred = find_best_block_list(raw)
    if inferred:
        return inferred
    raise ValueError("Input JSON must contain text-like or table-like blocks.")

def normalize_aliases(block, index):
    normalized = dict(block)
    for canonical, aliases in ALIASES.items():
        if normalized.get(canonical) is not None:
            continue
        for alias in aliases:
            if normalized.get(alias) is not None:
                normalized[canonical] = normalized[alias]
                break
    if not normalized.get("block_id"):
        page = normalized.get("page_number") or normalized.get("page") or "unknown"
        seed = normalized.get("text") or normalized.get("content") or "block"
        normalized["block_id"] = f"generated-p{page}-{index:04d}-{slugify(str(seed)[:32])}"
        normalized["generated_block_id"] = True
    if not normalized.get("reading_order"):
        normalized["reading_order"] = index
    return normalized

def norm_type(block):
    return str(block.get("block_type_guess") or "paragraph").strip().lower().replace("_", "-")

def is_uppercase_heading(text):
    letters = [char for char in text if char.isalpha()]
    return bool(letters) and sum(char.isupper() for char in letters) / len(letters) >= 0.8

def contains_keyword(text):
    lowered = text.lower()
    return any(keyword in lowered for keyword in SECTION_KEYWORDS)

def classify_heading(block, baseline):
    text = str(block.get("text") or "").strip()
    guess = norm_type(block)
    if not text:
        return None
    if guess in {"heading", "title", "section-heading", "h1", "h2", "h3"}:
        if guess == "h3":
            return 3
        if guess == "h2" or contains_keyword(text):
            return 2
        return 1 if guess == "title" else 2
    words = re.findall(r"\w+", text)
    short_text = len(words) <= 12
    font_size = float(block.get("font_size") or baseline)
    bold = bool(block.get("is_bold"))
    uppercase = is_uppercase_heading(text)
    keyword = contains_keyword(text) or bool(block.get("section_hint"))
    profile = globals().get("semantic_profile", {})
    heading_font_size = profile.get("heading_font_size", baseline + 2)
    h1_font_size = profile.get("h1_font_size", baseline + 5)
    large = font_size >= heading_font_size
    very_large = font_size >= h1_font_size
    if short_text and (very_large or (large and bold) or (bold and uppercase) or (bold and keyword)):
        if very_large or (uppercase and font_size >= baseline + 3):
            return 1
        if keyword or large:
            return 2
        return 3
    return None

def classify_block(block, baseline):
    level = classify_heading(block, baseline)
    if level:
        return "heading", level
    text = str(block.get("text") or "")
    guess = norm_type(block)
    if guess in {"table", "table-candidate"}:
        return "table", None
    if re.match(r"^\s*(?:[-*•]|[a-zA-Z]\)|\d+[.)])\s+", text):
        return ("ordered-list" if re.match(r"^\s*\d+[.)]\s+", text) else "unordered-list"), None
    confidence = block.get("confidence")
    profile = globals().get("semantic_profile", {})
    low_confidence_threshold = profile.get("low_confidence_threshold", 0.45)
    if confidence is not None and float(confidence) < low_confidence_threshold:
        return "aside", None
    return "paragraph", None

def percentile(values, pct):
    if not values:
        return 0.0
    ordered = sorted(values)
    index = round((len(ordered) - 1) * pct)
    return float(ordered[index])

def learn_semantic_profile(blocks, source_name="uploaded-json"):
    font_sizes = [float(block["font_size"]) for block in blocks if block.get("font_size")]
    confidences = [float(block["confidence"]) for block in blocks if block.get("confidence") is not None]
    heading_sizes = [float(block["font_size"]) for block in blocks if block.get("font_size") and norm_type(block) in {"heading", "title", "h1", "h2", "h3"}]
    body_font = float(median(font_sizes)) if font_sizes else 11.0
    heading_font = float(median(heading_sizes)) if heading_sizes else max(body_font + 2.0, 13.0)
    h1_font = max(percentile(font_sizes, 0.9), heading_font + 1.0) if font_sizes else 16.0
    low_confidence = min(0.6, max(0.45, percentile(confidences, 0.1))) if confidences else 0.45
    return {
        "source_name": source_name,
        "block_count": len(blocks),
        "body_font_size": round(body_font, 3),
        "heading_font_size": round(heading_font, 3),
        "h1_font_size": round(h1_font, 3),
        "low_confidence_threshold": round(low_confidence, 3),
        "learned_notes": [
            "Learned from prepared JSON metadata, not PDF parsing.",
            "Saved thresholds can be reused for future similar reports.",
        ],
    }

def slugify(value):
    return re.sub(r"[^a-zA-Z0-9]+", "-", str(value).strip().lower()).strip("-") or "block"

def bbox_value(block):
    if block.get("bbox") is not None:
        return str(block.get("bbox"))
    coords = [block.get("x0"), block.get("y0"), block.get("x1"), block.get("y1")]
    return ",".join(str(c) for c in coords) if all(c is not None for c in coords) else ""

def block_label(block, fallback="content block"):
    text = " ".join(str(block.get("text") or "").split())
    return text[:96] if text else fallback

def attrs(block, tag, suffix=None, semantic_role=None, label=None, extra=None):
    element_id = f"{tag}-{slugify(block.get('block_id'))}"
    if suffix is not None:
        element_id = f"{element_id}-{slugify(str(suffix))}"
    values = {
        "id": element_id,
        "data-page": "" if block.get("page_number") is None else str(block.get("page_number")),
        "data-source-block": str(block.get("block_id")),
        "data-block-type": str(block.get("block_type_guess") or tag),
        "data-bbox": bbox_value(block),
        "data-confidence": "" if block.get("confidence") is None else f"{float(block.get('confidence')):.3f}",
        "data-reading-order": "" if block.get("reading_order") is None else str(block.get("reading_order")),
        "data-semantic-role": str(semantic_role or tag),
        "data-source-file": str(block.get("source_file") or ""),
        "data-font-size": "" if block.get("font_size") is None else str(block.get("font_size")),
        "data-font-name": str(block.get("font_name") or ""),
        "data-generated-block-id": str(bool(block.get("generated_block_id"))).lower(),
    }
    if label:
        values["aria-label"] = label
    if extra:
        values.update({key: "" if value is None else str(value) for key, value in extra.items()})
    return " ".join(f'{k}="{html.escape(v, quote=True)}"' for k, v in values.items())

def parse_table(block):
    rows = block.get("rows") or block.get("table")
    if rows:
        return [[str(cell).strip() for cell in row] for row in rows]
    lines = [line.strip() for line in str(block.get("text") or "").splitlines() if line.strip()]
    if len(lines) > 1 and all("," in line for line in lines):
        parsed = [line.split(",") for line in lines]
        if all(len(row) == len(parsed[0]) for row in parsed):
            return [[cell.strip() for cell in row] for row in parsed]
    return None

def render_block(block, kind, level):
    text = html.escape(str(block.get("text") or ""))
    if kind == "heading":
        level = min(max(level or 2, 1), 3)
        heading_id = f"h{level}-{slugify(block.get('block_id'))}"
        section_attrs = attrs(block, "section", semantic_role="section", label=block_label(block, "section"), extra={"aria-labelledby": heading_id, "data-section-level": level, "data-section-title": block_label(block, "section")})
        heading_attrs = attrs(block, f"h{level}", semantic_role="heading", extra={"data-heading-level": level})
        return f'<section {section_attrs}><h{level} {heading_attrs}>{text}</h{level}></section>'
    if kind == "table":
        rows = parse_table(block)
        caption_id = f"caption-{slugify(block.get('block_id'))}"
        if not rows:
            figure_attrs = attrs(block, "figure", semantic_role="unclear-table", label=block_label(block, "table"))
            return f'<figure {figure_attrs}><figcaption id="{caption_id}">Unclear table-like block preserved as text.</figcaption><pre>{text}</pre></figure>'
        header, body = rows[0], rows[1:]
        head = "".join(f"<th {attrs(block, 'th', suffix=i, semantic_role='table-header-cell', extra={'scope': 'col', 'data-column-index': i})}>{html.escape(cell)}</th>" for i, cell in enumerate(header, start=1))
        body_html = ""
        for row_index, row in enumerate(body, start=1):
            cells = "".join(f"<td {attrs(block, 'td', suffix=f'{row_index}-{column_index}', semantic_role='table-data-cell', extra={'data-row-index': row_index, 'data-column-index': column_index})}>{html.escape(cell)}</td>" for column_index, cell in enumerate(row, start=1))
            body_html += f"<tr>{cells}</tr>"
        table_attrs = attrs(block, "table", semantic_role="table", extra={"aria-describedby": caption_id})
        return f'<table {table_attrs}><caption id="{caption_id}">{html.escape(block_label(block, "table"))}</caption><thead><tr>{head}</tr></thead><tbody>{body_html}</tbody></table>'
    if kind in {"unordered-list", "ordered-list"}:
        tag = "ol" if kind == "ordered-list" else "ul"
        items = [line.strip().lstrip("-*• ") for line in str(block.get("text") or "").splitlines() if line.strip()]
        item_html = "".join(f"<li {attrs(block, 'li', suffix=index, semantic_role='list-item', extra={'data-list-item-index': index})}>{html.escape(item)}</li>" for index, item in enumerate(items, start=1))
        return f'<{tag} {attrs(block, tag, semantic_role=kind, label=block_label(block, "list"))}>' + item_html + f'</{tag}>'
    if kind == "aside":
        return f'<aside {attrs(block, "aside", semantic_role="low-confidence-note", label=block_label(block))}>{text}</aside>'
    return f'<p {attrs(block, "p", semantic_role="paragraph", label=block_label(block))}>{text}</p>'

def render_outline(blocks):
    items = []
    for block in blocks:
        kind, level = classify_block(block, baseline)
        if kind == "heading":
            level = min(max(level or 2, 1), 3)
            heading_id = f"h{level}-{slugify(block.get('block_id'))}"
            items.append(f'<li><a href="#{heading_id}" data-source-block="{html.escape(str(block.get("block_id")))}">{html.escape(block_label(block, "Untitled section"))}</a></li>')
    return '<nav id="document-outline" aria-label="Document outline"><ol>' + ''.join(items) + '</ol></nav>' if items else ''

## Built-in Hard JSON Smoke Tests

Run this before uploading your own file during the presentation. It checks annual-report-like cases: nested page JSON, missing block IDs, alternate field names, bullet/numbered lists, clear tables, unclear table candidates, and low-confidence OCR notes.

In [ ]:
SMOKE_CASES = {
    "minimal_missing_ids": [
        {"text": "PUBLICATIONS", "font_size": 19, "is_bold": True, "confidence": 0.96},
        {"text": "The faculty published journal articles and conference papers.", "confidence": 0.85},
        {"text": "• Journal articles: 45\n• Conference papers: 62", "confidence": 0.82},
        {"text": "possibly unreadable handwritten annotation", "confidence": 0.18},
    ],
    "nested_annual_report": {
        "document": {
            "pages": [
                {
                    "pageNumber": 4,
                    "elements": [
                        {"content": "DEPARTMENT OF COMPUTER SCIENCE", "page": 4, "fontSize": 20, "bold": True, "role": "heading", "probability": 0.99},
                        {"content": "Research Grants and Collaboration", "page": 4, "fontSize": 15, "bold": True, "section": "Grants", "probability": 0.94},
                        {"content": "1. DST project continued\n2. Industry collaboration signed", "page": 4, "probability": 0.91},
                        {"content": "Grant Scheme | Amount | PI\nSERB | 1250000 | Dr. Rao", "page": 5, "label": "table-candidate", "probability": 0.87},
                        {"content": "smudged footer possible duplicate text", "page": 5, "probability": 0.21},
                    ],
                }
            ]
        }
    },
    "table_and_outreach": {
        "content_blocks": [
            {"uid": "awards-heading", "raw_text": "FACULTY AWARDS", "font_size": 17, "is_bold": True, "category": "title", "confidence": 0.97},
            {"uid": "awards-table-clear", "block_type": "table", "table_rows": [["Name", "Award"], ["Dr. Iyer", "Best Researcher"]], "confidence": 0.93},
            {"uid": "awards-table-unclear", "value": "Name Award Year maybe columns broken", "block_type": "table-candidate", "confidence": 0.52},
            {"uid": "outreach-list", "value": "- Workshop for schools\n- Conference tutorial", "confidence": 0.89},
        ]
    },
}


def semantify_raw_for_smoke(raw, name):
    blocks = [normalize_aliases(block, index) for index, block in enumerate(extract_blocks(raw), start=1)]
    blocks = sorted(blocks, key=lambda b: (b.get("reading_order") is None, b.get("reading_order") or 0))
    global semantic_profile
    semantic_profile = learn_semantic_profile(blocks, source_name=name)
    baseline = semantic_profile["body_font_size"]
    globals()["baseline"] = baseline
    body = []
    for block in blocks:
        kind, level = classify_block(block, baseline)
        body.append(render_block(block, kind, level))
    outline_html = render_outline(blocks)
    report_html = """<!doctype html><html lang="en"><head><meta charset="utf-8"><title>Smoke</title></head><body><header>{}</header><main id="report-content" aria-label="Semantic annual report content"><article id="semantic-report">{}</article></main></body></html>""".format(outline_html, "\n".join(body))
    soup = BeautifulSoup(report_html, "lxml")
    elements = soup.find("main").find_all({"section", "h1", "h2", "h3", "p", "aside", "ul", "ol", "li", "table", "th", "td", "figure"})
    traceable = [element for element in elements if element.get("id") and element.get("data-source-block") and element.get("data-semantic-role")]
    return {
        "name": name,
        "ok": len(traceable) >= len(blocks),
        "blocks": len(blocks),
        "traceable_elements": len(traceable),
        "profile": semantic_profile,
    }

smoke_results = [semantify_raw_for_smoke(raw, name) for name, raw in SMOKE_CASES.items()]
assert all(result["ok"] for result in smoke_results), smoke_results
smoke_results

In [ ]:
raw = json.loads(Path(input_filename).read_text(encoding="utf-8"))
blocks = [normalize_aliases(block, index) for index, block in enumerate(extract_blocks(raw), start=1)]
blocks = sorted(blocks, key=lambda b: (b.get("reading_order") is None, b.get("reading_order") or 0))
sizes = [float(block["font_size"]) for block in blocks if block.get("font_size")]
baseline = median(sizes) if sizes else 11.0
semantic_profile = learn_semantic_profile(blocks, source_name=input_filename)
baseline = semantic_profile["body_font_size"]

body = []
for block in blocks:
    if not block.get("block_id"):
        raise ValueError("Every block must include block_id for traceability.")
    kind, level = classify_block(block, baseline)
    body.append(render_block(block, kind, level))

outline_html = render_outline(blocks)
report_html = """<!doctype html>
<html lang="en">
<head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1"><title>RAISE Semantic HTML Report</title></head>
<body><header role="banner"><h1>RAISE HTML5 Semantification Output</h1>{}</header><main id="report-content" role="main" aria-label="Semantic annual report content"><article id="semantic-report" aria-label="Semantified annual report">
{}
</article></main><footer role="contentinfo"><p>Module owner: Siddharth Tripathi. Next stage input variable: <code>semantic_html_path</code>.</p></footer></body></html>""".format(outline_html, "\n".join(body))

Path("report.html").write_text(report_html, encoding="utf-8")

soup = BeautifulSoup(report_html, "lxml")
traceable_tags = {"section", "h1", "h2", "h3", "p", "aside", "ul", "ol", "li", "table", "th", "td", "figure"}
main = soup.find("main")
elements = main.find_all(traceable_tags) if main else []
issues = []
for element in elements:
    if not element.get("id") or not element.get("data-source-block") or not element.get("data-semantic-role"):
        issues.append({"severity": "error", "message": f"Missing traceability on <{element.name}>"})

validation_report = {
    "ok": not any(issue["severity"] == "error" for issue in issues),
    "input_block_count": len(blocks),
    "html_element_count": len(elements),
    "traceable_element_count": sum(1 for element in elements if element.get("id") and element.get("data-source-block")),
    "issues": issues,
}
Path("validation_report.json").write_text(json.dumps(validation_report, indent=2), encoding="utf-8")
Path("semantic_profile.json").write_text(json.dumps(semantic_profile, indent=2), encoding="utf-8")
validation_report

In [ ]:
from IPython.display import IFrame, display

print("Validation report:")
print(json.dumps(validation_report, indent=2))

print("Semantic profile:")
print(json.dumps(semantic_profile, indent=2))

display(IFrame("report.html", width="100%", height=650))

In [ ]:
semantic_html_path = "report.html"
validation_report_path = "validation_report.json"
semantic_profile_path = "semantic_profile.json"

files.download(semantic_html_path)
files.download(validation_report_path)
files.download(semantic_profile_path)